<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.6-observability-with-langfuse/notebooks/exercises-5_6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.6: Observability: Langfuse & LangSmith

*Module 5 · Production Applications with GenAI APIs*

> Your AI app is live with 1,000 users. Something goes wrong. Without observability, you're blind — no idea which calls failed, why latency spiked, or where money is leaking. Langfuse and LangSmith give you X-ray vision into every LLM call.

---

## About this notebook

This notebook contains the **6 runnable code examples** from the published lesson `lesson-5.6.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Langfuse — Open-Source LLM Observability — `part1a_langfuse_setup.py`
2. Step 1 — Langfuse — Open-Source LLM Observability — `part1b_langfuse_decorator.py`
3. Step 2 — LangSmith — The LangChain Ecosystem Tool — `part2_langsmith.py`
4. Step 3 — Quality Scoring — Was the Answer Actually Good? — `part3_quality_scoring.py`
5. Step 4 — Dashboards — The Metrics That Matter — `part4_metrics_exporter.py`
6. Step 5 — Project: Fully Instrumented Chat App — `project_instrumented_chat.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai transformers langchain requests


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
#   export LANGCHAIN_API_KEY=sk-...
#   export LANGSMITH_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")
os.environ.setdefault("LANGCHAIN_API_KEY", "YOUR_LANGCHAIN_API_KEY_HERE")
os.environ.setdefault("LANGSMITH_API_KEY", "YOUR_LANGSMITH_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Langfuse — Open-Source LLM Observability

Free, self-hostable, works with any LLM. The most popular open-source option.

**`part1a_langfuse_setup.py`** · _Block 1 of 6_

LANGFUSE: Instrument your first LLM call — pip install langfuse google-generativeai


In [ ]:
# =============================================
# LANGFUSE: Instrument your first LLM call
# pip install langfuse google-generativeai
# Sign up free: https://cloud.langfuse.com
# =============================================

from langfuse import Langfuse
import google.generativeai as genai
import os, time

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Initialize Langfuse client
langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host="https://cloud.langfuse.com",  # or your self-hosted URL
)

# ── Trace a simple LLM call ──
def ask_with_tracing(question: str, user_id: str = "anonymous") -> str:
    """Make an LLM call with full Langfuse tracing."""
    
    # Create a trace (one per user interaction)
    trace = langfuse.trace(
        name="chat-response",
        user_id=user_id,
        metadata={"source": "web", "session": "demo"},
    )
    
    # Create a generation span (tracks the LLM call)
    generation = trace.generation(
        name="gemini-call",
        model="gemini-2.0-flash",
        input=question,
        model_parameters={"temperature": 0.3},
    )
    
    # Actually call the LLM
    start = time.time()
    model = genai.GenerativeModel("gemini-2.0-flash")
    response = model.generate_content(question)
    latency = time.time() - start
    
    answer = response.text
    
    # Complete the generation with output + metrics
    generation.end(
        output=answer,
        usage={
            "input": response.usage_metadata.prompt_token_count,
            "output": response.usage_metadata.candidates_token_count,
        },
        metadata={"latency_ms": round(latency * 1000)},
    )
    
    return answer

# ── Test it ──
answer = ask_with_tracing("What is RAG in 2 sentences?", user_id="student_001")
print(f"Answer: {answer}")
print("✅ Trace sent to Langfuse! Check your dashboard.")

# Flush to ensure all traces are sent
langfuse.flush()


**`part1b_langfuse_decorator.py`** · _Block 2 of 6_

LANGFUSE DECORATOR: Zero-code instrumentation — Just add @observe() to any function!


In [ ]:
# =============================================
# LANGFUSE DECORATOR: Zero-code instrumentation
# Just add @observe() to any function!
# =============================================

from langfuse.decorators import observe, langfuse_context

@observe()
def process_query(question: str) -> str:
    """Full pipeline: embed → retrieve → generate. All auto-traced!"""
    
    # Step 1: Embed the question
    embedding = embed_question(question)
    
    # Step 2: Retrieve relevant docs
    docs = retrieve_documents(embedding)
    
    # Step 3: Generate response
    answer = generate_response(question, docs)
    
    # Attach metadata to the trace
    langfuse_context.update_current_observation(
        metadata={"num_docs": len(docs), "pipeline": "rag_v2"}
    )
    
    return answer

@observe()
def embed_question(text: str) -> list:
    """Embed a question — auto-traced as a span."""
    result = genai.embed_content(model="models/text-embedding-004", content=text)
    return result["embedding"]

@observe()
def retrieve_documents(embedding: list) -> list[str]:
    """Retrieve docs — auto-traced as a span."""
    # In production: query vector database
    return ["RAG combines retrieval with generation...", "Vector databases store embeddings..."]

@observe(as_type="generation")
def generate_response(question: str, docs: list[str]) -> str:
    """Generate answer — auto-traced as a GENERATION (tracks tokens+cost)."""
    context = "\n".join(docs)
    prompt = f"Context: {context}\n\nQuestion: {question}\nAnswer:"
    
    model = genai.GenerativeModel("gemini-2.0-flash")
    response = model.generate_content(prompt)
    
    # Update generation with model info
    langfuse_context.update_current_observation(
        model="gemini-2.0-flash",
        input=prompt,
        output=response.text,
        usage={"input": response.usage_metadata.prompt_token_count,
               "output": response.usage_metadata.candidates_token_count},
    )
    return response.text

# ── Run the pipeline — all steps auto-traced! ──
result = process_query("How does RAG work?")
print(f"Answer: {result[:100]}...")

print("""
What Langfuse now shows in the dashboard:
  📊 Trace: "process_query"
    ├─ Span: "embed_question" (latency: 120ms)
    ├─ Span: "retrieve_documents" (latency: 45ms) 
    └─ Generation: "generate_response"
         ├─ Model: gemini-2.0-flash
         ├─ Input tokens: 156
         ├─ Output tokens: 89
         ├─ Latency: 1,230ms
         └─ Cost: $0.00004
""")


> **What just happened?** Two instrumentation approaches: (1) Manual tracing — create trace → create generation → end with output/usage. Full control over what's tracked. (2) Decorator-based — just add @observe() to any function. Langfuse automatically creates spans for each function call and nests them. The as_type="generation" decorator specifically tracks LLM calls with tokens and cost. Both approaches show the same result in the dashboard: a timeline of every step with latency, tokens, and cost.


### Step 2 · LangSmith — The LangChain Ecosystem Tool

Built by the LangChain team. Deep integration with LangChain and LangGraph.

**`part2_langsmith.py`** · _Block 3 of 6_

LANGSMITH: Tracing with the LangChain ecosystem — pip install langsmith


In [ ]:
# =============================================
# LANGSMITH: Tracing with the LangChain ecosystem
# pip install langsmith
# Sign up: https://smith.langchain.com
# =============================================

import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "netsetos-genai-course"

from langsmith import traceable
from langsmith.run_helpers import get_current_run_tree

# ── @traceable: auto-traces any function ──

@traceable(name="rag_pipeline")
def rag_pipeline(question: str) -> str:
    """Full RAG pipeline — every step auto-traced."""
    docs = retrieve(question)
    answer = generate(question, docs)
    return answer

@traceable(name="retrieve")
def retrieve(query: str) -> list[str]:
    # In production: vector DB query
    return ["RAG retrieves relevant context before generating..."]

@traceable(name="generate", run_type="llm")
def generate(question: str, docs: list[str]) -> str:
    model = genai.GenerativeModel("gemini-2.0-flash")
    context = "\n".join(docs)
    response = model.generate_content(f"Context: {context}\n\nQ: {question}\nA:")
    return response.text

# Run it — automatically traced in LangSmith!
answer = rag_pipeline("What is vector search?")
print(f"Answer: {answer[:100]}...")
print("✅ Trace visible in LangSmith dashboard")


### Step 3 · Quality Scoring — Was the Answer Actually Good?

Latency and cost don't tell you if the answer was helpful. Scoring does.

**`part3_quality_scoring.py`** · _Block 4 of 6_

QUALITY SCORING: 3 ways to score outputs — 1. User feedback (thumbs up/down)


In [ ]:
# =============================================
# QUALITY SCORING: 3 ways to score outputs
# 1. User feedback (thumbs up/down)
# 2. LLM-as-Judge (automatic)
# 3. Manual review (human expert)
# =============================================

class QualityScorer:
    """Score LLM outputs and attach scores to Langfuse traces."""
    
    def __init__(self):
        self.langfuse = Langfuse()
        self.judge = genai.GenerativeModel("gemini-2.0-flash",
            generation_config={"temperature": 0.0})
    
    # ── Method 1: User feedback ──
    def score_user_feedback(self, trace_id: str, thumbs_up: bool):
        """Record whether the user liked the response."""
        self.langfuse.score(
            trace_id=trace_id,
            name="user-feedback",
            value=1 if thumbs_up else 0,
            comment="thumbs up" if thumbs_up else "thumbs down",
        )
    
    # ── Method 2: LLM-as-Judge (automatic) ──
    def score_llm_judge(self, trace_id: str, question: str, answer: str) -> float:
        """Use a cheap LLM to score the answer quality 0-1."""
        prompt = f"""Rate this answer's quality from 0.0 to 1.0.
Question: {question[:200]}
Answer: {answer[:500]}
Criteria: accuracy, completeness, clarity.
Reply with ONLY a number."""
        
        resp = self.judge.generate_content(prompt)
        try:
            score = float(resp.text.strip())
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        
        self.langfuse.score(
            trace_id=trace_id,
            name="llm-judge",
            value=score,
            comment=f"Auto-scored by gemini-2.0-flash",
        )
        return score
    
    # ── Method 3: Specific quality checks ──
    def score_hallucination_check(self, trace_id: str, answer: str, context: str) -> float:
        """Check if the answer is grounded in the provided context."""
        prompt = f"""Is this answer FULLY supported by the context? Score 0.0-1.0.
0.0 = completely hallucinated (makes up facts not in context)
1.0 = fully grounded (every claim is in the context)

Context: {context[:500]}
Answer: {answer[:500]}
Score (0.0-1.0):"""
        
        resp = self.judge.generate_content(prompt)
        try:
            score = float(resp.text.strip())
        except:
            score = 0.5
        
        self.langfuse.score(trace_id=trace_id, name="groundedness", value=score)
        return score

# ── Demo: Score a traced call ──
scorer = QualityScorer()

# Make a traced call
trace = langfuse.trace(name="scored-chat", user_id="student_001")
question = "What is the difference between RAG and fine-tuning?"
answer = ask_with_tracing(question)

# Score it 3 ways
scorer.score_user_feedback(trace.id, thumbs_up=True)
quality = scorer.score_llm_judge(trace.id, question, answer)
print(f"LLM judge score: {quality:.2f}")

print("""
In the Langfuse dashboard, this trace now shows:
  📊 Trace: "scored-chat"
    ├─ Generation: gemini-2.0-flash (tokens, cost, latency)
    ├─ Score: user-feedback = 1 (thumbs up)
    ├─ Score: llm-judge = 0.85 (auto quality score)
    └─ Score: groundedness = 0.92 (hallucination check)
    
Over thousands of traces, you can:
  • Track average quality score over time
  • Compare quality across different prompts/models
  • Find traces where quality dropped (and investigate why)
  • Correlate user feedback with LLM judge scores
""")


> **What just happened?** Three scoring methods attached to traces: (1) User feedback — simple thumbs up/down from the UI. (2) LLM judge — automatic quality score (0-1) using a cheap model as evaluator. (3) Groundedness check — verifies the answer is supported by the context (catches hallucinations). All scores appear on the trace in the dashboard, enabling you to track quality trends over time, compare prompts, and investigate drops.


### Step 4 · Dashboards — The Metrics That Matter

Turn raw traces into actionable insights: cost trends, latency percentiles, quality over time.

**`part4_metrics_exporter.py`** · _Block 5 of 6_

METRICS EXPORTER: Extract key metrics from — traces for dashboards and alerts.


In [ ]:
# =============================================
# METRICS EXPORTER: Extract key metrics from
# traces for dashboards and alerts.
# =============================================

from collections import defaultdict
from datetime import datetime, timedelta
import statistics

class ObservabilityDashboard:
    """Track and report key LLM metrics."""
    
    def __init__(self):
        self.traces = []
    
    def record(self, trace_data: dict):
        """Record a trace for local analytics."""
        trace_data["timestamp"] = datetime.now()
        self.traces.append(trace_data)
    
    def report(self, hours: int = 24):
        """Generate a comprehensive dashboard report."""
        cutoff = datetime.now() - timedelta(hours=hours)
        recent = [t for t in self.traces if t["timestamp"] >= cutoff]
        
        if not recent:
            print("No traces in the last {hours} hours.")
            return
        
        print(f"\n📊 Observability Dashboard (last {hours}h)")
        print("═" * 55)
        
        # Volume
        print(f"\n  Total requests:  {len(recent)}")
        errors = [t for t in recent if t.get("error")]
        print(f"  Error rate:      {len(errors)/len(recent):.1%}")
        
        # Latency
        latencies = [t["latency_ms"] for t in recent if "latency_ms" in t]
        if latencies:
            print(f"\n  Latency (ms):")
            print(f"    P50:  {statistics.median(latencies):>6.0f}ms")
            print(f"    P95:  {sorted(latencies)[int(len(latencies)*0.95)]:>6.0f}ms")
            print(f"    P99:  {sorted(latencies)[int(len(latencies)*0.99)]:>6.0f}ms")
        
        # Cost
        costs = [t["cost_usd"] for t in recent if "cost_usd" in t]
        if costs:
            print(f"\n  Cost:")
            print(f"    Total:    ${sum(costs):.4f}")
            print(f"    Avg/call: ${statistics.mean(costs):.5f}")
            print(f"    Projected monthly: ${sum(costs)/hours*720:.2f}")
        
        # Quality
        scores = [t["quality_score"] for t in recent if "quality_score" in t]
        if scores:
            print(f"\n  Quality:")
            print(f"    Avg score:    {statistics.mean(scores):.2f}")
            low_quality = [s for s in scores if s < 0.6]
            print(f"    Low quality:  {len(low_quality)} ({len(low_quality)/len(scores):.0%})")
        
        # Per model breakdown
        by_model = defaultdict(lambda: {"count": 0, "cost": 0, "latency": []})
        for t in recent:
            m = by_model[t.get("model", "unknown")]
            m["count"] += 1
            m["cost"] += t.get("cost_usd", 0)
            if "latency_ms" in t:
                m["latency"].append(t["latency_ms"])
        
        print(f"\n  By model:")
        for model, data in by_model.items():
            avg_lat = statistics.mean(data["latency"]) if data["latency"] else 0
            print(f"    {model:25s}  {data['count']:>4} calls  ${data['cost']:>7.4f}  {avg_lat:>5.0f}ms avg")

# ── Simulate metrics collection ──
dash = ObservabilityDashboard()

# Simulate 20 traced calls
import random
for i in range(20):
    dash.record({
        "model": random.choice(["gemini-2.0-flash", "gpt-4o-mini"]),
        "latency_ms": random.randint(200, 3000),
        "cost_usd": random.uniform(0.0001, 0.005),
        "quality_score": random.uniform(0.5, 1.0),
        "error": random.random() < 0.05,  # 5% error rate
    })

dash.report(hours=24)


> **What just happened?** We built an ObservabilityDashboard that computes the 5 key metrics from traces: (1) Volume + error rate, (2) Latency percentiles (P50, P95, P99), (3) Cost (total, per-call average, projected monthly), (4) Quality (average score, low-quality percentage), (5) Per-model breakdown. These are the exact metrics every production AI team monitors. In real production, Langfuse and LangSmith provide pre-built dashboards with these metrics — our code replicates the analytics for understanding.


### Step 5 · Project: Fully Instrumented Chat App

Combine everything: Langfuse tracing + quality scoring + metrics + alerts.

**`project_instrumented_chat.py`** · _Block 6 of 6_

FULLY INSTRUMENTED CHAT APPLICATION — Every call traced, scored, and monitored.


In [ ]:
# =============================================
# FULLY INSTRUMENTED CHAT APPLICATION
# Every call traced, scored, and monitored.
# =============================================

class InstrumentedChat:
    """Chat app with end-to-end observability."""
    
    def __init__(self):
        self.langfuse = Langfuse()
        self.scorer = QualityScorer()
        self.dashboard = ObservabilityDashboard()
        self.model = genai.GenerativeModel("gemini-2.0-flash",
            system_instruction="You are a helpful Netsetos tutor.")
    
    def chat(self, message: str, user_id: str = "anon", session_id: str = "default") -> dict:
        """Send a message with full observability."""
        
        # Create trace
        trace = self.langfuse.trace(
            name="chat",
            user_id=user_id,
            session_id=session_id,
            metadata={"source": "api"},
        )
        
        # Track the LLM generation
        gen = trace.generation(
            name="generate",
            model="gemini-2.0-flash",
            input=message,
        )
        
        # Call LLM
        start = time.time()
        try:
            response = self.model.generate_content(message)
            answer = response.text
            latency = (time.time() - start) * 1000
            
            in_tok = response.usage_metadata.prompt_token_count
            out_tok = response.usage_metadata.candidates_token_count
            cost = (in_tok * 0.075 + out_tok * 0.30) / 1_000_000
            
            gen.end(output=answer, usage={"input": in_tok, "output": out_tok})
            error = False
        
        except Exception as e:
            answer = f"Error: {e}"
            latency = (time.time() - start) * 1000
            cost = 0
            gen.end(output=answer, level="ERROR", status_message=str(e))
            error = True
        
        # Auto-score quality
        quality = 0.0
        if not error:
            quality = self.scorer.score_llm_judge(trace.id, message, answer)
        
        # Record for dashboard
        self.dashboard.record({
            "model": "gemini-2.0-flash", "latency_ms": latency,
            "cost_usd": cost, "quality_score": quality, "error": error,
            "user_id": user_id,
        })
        
        # Alert on quality drops
        if quality < 0.5 and not error:
            print(f"  ⚠️ LOW QUALITY ALERT: score={quality:.2f} for '{message[:50]}'")
        
        return {"answer": answer, "latency_ms": latency, "cost": cost,
                "quality": quality, "trace_id": trace.id}

# ── Run a simulated workload ──
chat = InstrumentedChat()

queries = [
    ("What is RAG?", "student_001"),
    ("Explain transformers", "student_002"),
    ("How do embeddings work?", "student_001"),
    ("What is fine-tuning vs RAG?", "student_003"),
    ("Explain attention mechanism", "student_002"),
]

print("Running instrumented chat:\n")
for query, user in queries:
    result = chat.chat(query, user_id=user)
    print(f"  [{user}] Q: {query}")
    print(f"           A: {result['answer'][:80]}...")
    print(f"           ⏱ {result['latency_ms']:.0f}ms  💰${result['cost']:.5f}  ⭐{result['quality']:.2f}\n")

# Show the dashboard
chat.dashboard.report()
langfuse.flush()

print("""
Every single call now has:
  ✅ Full trace in Langfuse (input, output, tokens, cost)
  ✅ Automatic quality score (LLM judge)
  ✅ User/session tracking (per-user analytics)
  ✅ Local metrics dashboard (latency, cost, quality)
  ✅ Alerts on quality drops (<0.5)
  ✅ Error tracking with stack traces

This is production-grade observability.
""")


> **What just happened?** We built a fully instrumented chat application where every single LLM call is: (1) Traced in Langfuse with input, output, tokens, and cost. (2) Auto-scored for quality using LLM-as-Judge. (3) Monitored with local dashboard metrics (latency P50/P95/P99, cost, quality trends). (4) Alerted when quality drops below threshold. Each response includes a trace_id that links to the full trace in the Langfuse UI. This is how production AI teams operate: every call is visible, measurable, and debuggable.


---

## Wrap-up

You've walked through all 6 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
